# Lesson 5: Urban Form and Buildings

**Planning question: How dense is the urban fabric, and what shape is it?**

Building footprints tell us a lot about a place: how much land is built on, how large the plots are,
and whether the buildings follow a regular pattern. This is **urban morphology**.

In this notebook you will compare building footprints in **Jayanagar** (planned) and **Shivajinagar** (dense and organic).

Part of [RS & GIS for Spatial Planners](https://soniadas123.github.io/rs-gis-spatial-planner/).

In [ ]:
!pip install -q osmnx momepy

In [ ]:
import osmnx as ox
import momepy
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import box

In [ ]:
PLACES = {
    "Jayanagar": (12.9300, 77.5838),
    "Shivajinagar": (12.9857, 77.6050),
}
DIST = 700               # metres around each centre
UTM_CRS = "EPSG:32643"
CELL = 100               # grid cell size in metres for the density map

## Step 1: Download building footprints

We keep only polygons, project them to metres and fix any broken outlines.

In [ ]:
buildings = {}
for name, center in PLACES.items():
    b = ox.features_from_point(center, tags={"building": True}, dist=DIST)
    b = b[b.geom_type.isin(["Polygon", "MultiPolygon"])].to_crs(UTM_CRS)
    b["geometry"] = b.buffer(0)   # fixes small drawing errors in some outlines
    buildings[name] = b[["building", "geometry"]].reset_index(drop=True)
    print(name, "buildings:", len(b))

## Step 2: Figure-ground maps

The **figure-ground** drawing is a classic tool in urban design: buildings are black and open space is white.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, (name, b) in zip(axes, buildings.items()):
    b.plot(ax=ax, color="black")
    ax.set_title(name, fontsize=16)
    ax.set_axis_off()
plt.show()

## Step 3: Measure every building

For each building we calculate:

- **Footprint area** in square metres (GeoPandas)
- **Circular compactness**: how close the shape is to a circle, from 0 to 1 (momepy)
- **Elongation**: 1 is square, and values near 0 mean long and thin (momepy)
- **Orientation**: the angle of the building, from 0 to 45 degrees (momepy)

In [ ]:
for name, b in buildings.items():
    b["area_m2"] = b.area
    b["compactness"] = momepy.circular_compactness(b)
    b["elongation"] = momepy.elongation(b)
    b["orientation"] = momepy.orientation(b)

buildings["Jayanagar"].head()

## Step 4: Compare the two places

In [ ]:
rows = []
for name, b in buildings.items():
    rows.append({
        "Place": name,
        "Buildings": len(b),
        "Median footprint (m2)": round(b["area_m2"].median()),
        "Mean compactness": round(b["compactness"].mean(), 2),
        "Mean elongation": round(b["elongation"].mean(), 2),
        "Orientation spread (std, degrees)": round(b["orientation"].std(), 1),
    })
pd.DataFrame(rows).set_index("Place")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, b in buildings.items():
    ax.hist(b["area_m2"].clip(upper=600), bins=40, alpha=0.6, label=name)
ax.set_xlabel("Footprint area (m2, values above 600 grouped together)")
ax.set_ylabel("Number of buildings")
ax.legend()
plt.show()

## Step 5: Ground coverage on a grid

**Ground coverage** is the share of land covered by buildings. Planning rules often limit it, for example to 50 to 75 percent.
We build a grid of 100 m cells and work out the coverage of each cell.

In [ ]:
def make_grid(gdf, size):
    """Square grid covering the extent of gdf."""
    xmin, ymin, xmax, ymax = gdf.total_bounds
    cells = [box(x, y, x + size, y + size)
             for x in np.arange(xmin, xmax, size)
             for y in np.arange(ymin, ymax, size)]
    return gpd.GeoDataFrame(geometry=cells, crs=gdf.crs)


def ground_coverage(b, size):
    grid = make_grid(b, size)
    grid["cell_id"] = grid.index
    pieces = gpd.overlay(b[["geometry"]], grid, how="intersection")
    built = pieces.dissolve(by="cell_id").area
    grid["coverage_pct"] = (grid["cell_id"].map(built).fillna(0) / size ** 2 * 100).clip(upper=100)
    return grid


coverage = {name: ground_coverage(b, CELL) for name, b in buildings.items()}

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
for ax, (name, grid) in zip(axes, coverage.items()):
    grid.plot(ax=ax, column="coverage_pct", cmap="magma_r", vmin=0, vmax=80, legend=True,
              legend_kwds={"label": "Ground coverage (%)", "shrink": 0.7})
    buildings[name].boundary.plot(ax=ax, color="white", linewidth=0.2)
    ax.set_title(name, fontsize=16)
    ax.set_axis_off()
plt.show()

In [ ]:
for name, grid in coverage.items():
    print(f"{name}: average ground coverage {grid['coverage_pct'].mean():.1f}%")

## Step 6: Save

In [ ]:
for name, b in buildings.items():
    b.to_file("urban_form.gpkg", layer=f"{name.lower()}_buildings", driver="GPKG")
    coverage[name].to_file("urban_form.gpkg", layer=f"{name.lower()}_coverage", driver="GPKG")
print("Saved urban_form.gpkg")

## What this means for planning

- Coverage maps show where the fabric is already saturated and where there is room for infill.
- Building shape measures can help to sort a city into **morphological types**, such as plotted housing, informal dense fabric and large campuses.
- Footprints alone do not show height. Datasets with building heights, such as Google Open Buildings 2.5D, can add the third dimension so you can estimate floor area ratio.

**Limitations:** OSM building coverage in Bengaluru is good in central areas but patchy in places.
Check your area against satellite imagery before you draw conclusions.

## Try it yourself

1. Add an IT corridor such as Whitefield `(12.9698, 77.7500)` to `PLACES`.
2. Change `CELL` to 50 or 200 metres. How does the pattern change?
3. Look at the momepy documentation (https://docs.momepy.org) and try another measure, such as `momepy.convexity`.

---
*Enjoying the course? Support it with a [star on GitHub](https://github.com/soniadas123/rs-gis-spatial-planner).*